In [1]:
"""
===============================================================================
UNIFIED BCT PREDICTION & GSM OPTIMIZATION MODEL ENGINE (DUAL-ENGINE)
===============================================================================
Combines:
  1. Engine A: Empirical Power-Law Regression Model (Fitted: BCT = k * GSM^n)
  2. Engine B: First-Principles McKee Structural Model (Zero Data Fitting)
  3. Sealing Efficiency Uplift Integration (Glued vs. Taped)
  4. Target GSM Inverter & Paper/Cost Savings Calculator
  5. Board Structure Evaluator & Combinatorial Mill Grade Finder
===============================================================================
"""

'\n===============================================================================\nUNIFIED BCT PREDICTION & GSM OPTIMIZATION MODEL ENGINE (DUAL-ENGINE)\n===============================================================================\nCombines:\n  1. Engine A: Empirical Power-Law Regression Model (Fitted: BCT = k * GSM^n)\n  2. Engine B: First-Principles McKee Structural Model (Zero Data Fitting)\n  3. Sealing Efficiency Uplift Integration (Glued vs. Taped)\n  4. Target GSM Inverter & Paper/Cost Savings Calculator\n  5. Board Structure Evaluator & Combinatorial Mill Grade Finder\n===============================================================================\n'

In [2]:
import math
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass

In [3]:
# -----------------------------------------------------------------------------
# 1. DATA STRUCTURES & BENCHMARKS
# -----------------------------------------------------------------------------

@dataclass
class CartonSpec:
    idh_code: str
    length_mm: float
    width_mm: float
    height_mm: float
    ply_structure: List[int]       # [OuterLiner, Medium1, InnerLiner1, Medium2, InnerLiner2]
    total_gsm: int
    bf_outer: int
    bf_mediums: int
    measured_bct_kgf: float
    comment: str

In [4]:
BENCHMARK_SPECS = [
    CartonSpec(
        idh_code="1714527 / 2303677",
        length_mm=470, width_mm=280, height_mm=238,
        ply_structure=[300, 180, 300, 180, 300],
        total_gsm=1260, bf_outer=25, bf_mediums=20,
        measured_bct_kgf=800.0,
        comment="Original Spec Baseline (Min 800 Kgf)"
    ),
    CartonSpec(
        idh_code="2879212 / 2879211",
        length_mm=470, width_mm=280, height_mm=238,
        ply_structure=[230, 180, 230, 180, 230],
        total_gsm=1050, bf_outer=25, bf_mediums=20,
        measured_bct_kgf=600.0,
        comment="2023 First-Phase GSM Reduction"
    ),
    CartonSpec(
        idh_code="3147790 / 3158585",
        length_mm=470, width_mm=280, height_mm=238,
        ply_structure=[230, 140, 140, 140, 230],
        total_gsm=880, bf_outer=25, bf_mediums=20,
        measured_bct_kgf=480.0,
        comment="Optimized Spec (Tested for Noa Kalang)"
    )
]

In [5]:
SEALING_FACTORS = {
    "Taped Standard": 1.000,
    "Glued @ 15mm Deflection (Conservative)": 1.209,
    "Glued @ 7mm Deflection (Aggressive)": 1.413
}


In [6]:
# -----------------------------------------------------------------------------
# 2. DUAL PREDICTION ENGINES
# -----------------------------------------------------------------------------

class UnifiedBCTModelEngine:
    def __init__(self):
        # Empirical Power-Law Fitted Parameters (derived from log-linear fit)
        self.k = 0.03049704
        self.n = 1.423962
        self.ln_k = -3.490119
        self.r2 = 0.996079
        self.rmse = 8.26
        self.mape = 1.26

In [8]:
class UnifiedBCTModelEngine:
    def __init__(self):
        # Empirical Power-Law Fitted Parameters (derived from log-linear fit)
        self.k = 0.03049704
        self.n = 1.423962
        self.ln_k = -3.490119
        self.r2 = 0.996079
        self.rmse = 8.26
        self.mape = 1.26

    # -------------------------------------------------------------------------
    # Engine A: Empirical Power-Law Model
    # -------------------------------------------------------------------------
    def predict_empirical(self, total_gsm: float, phi: float = 1.00) -> float:
        """BCT = phi * k * (GSM)^n"""
        bct_base = self.k * math.pow(total_gsm, self.n)
        return round(bct_base * phi, 2)

    def invert_gsm_empirical(self, target_bct_kgf: float, phi: float = 1.00) -> float:
        """GSM = ( (target_bct / phi) / k ) ^ (1 / n)"""
        effective_bct = target_bct_kgf / phi
        gsm = math.pow(effective_bct / self.k, 1.0 / self.n)
        return round(gsm, 1)

    # -------------------------------------------------------------------------
    # Engine B: First-Principles McKee Model (Zero Data Fitting)
    # -------------------------------------------------------------------------
    def predict_first_principles(self,
                                 length_mm: float = 470,
                                 width_mm: float = 280,
                                 height_mm: float = 238,
                                 liners_gsm: List[float] = [230, 140, 230], # Outer, Center, Inner
                                 liners_bf: List[float] = [25, 20, 25],
                                 mediums_gsm: List[float] = [140, 140],
                                 mediums_bf: List[float] = [20, 20],
                                 flute_types: List[str] = ['C', 'B'],
                                 phi: float = 1.00) -> Dict[str, float]:
        """
        Pure physics calculation using TAPPI McKee formula & Component RCT assembly.
        No empirical BCT data fitting required.
        """
        # 1. Box Perimeter (m)
        perimeter_m = 2.0 * (length_mm + width_mm) / 1000.0

        # 2. Combined Board Caliper (m) - BC Flute combined ~ 6.3mm
        caliper_mm = 6.3
        caliper_m = caliper_mm / 1000.0

        # 3. Component Ring Crush Test (RCT) in N/m
        # Liner RCI factor = 0.06 * (BF^0.8) * 15000 N/m per gsm
        total_ect_n_m = 0.0

        for gsm, bf in zip(liners_gsm, liners_bf):
            rci = 0.06 * math.pow(bf, 0.8)
            rct = (gsm / 1000.0) * rci * 15000.0
            total_ect_n_m += rct

        # Medium RCT scaled by flute take-up (C=1.43, B=1.32)
        takeup_map = {'C': 1.43, 'B': 1.32, 'A': 1.54, 'E': 1.27}
        for gsm, bf, ftype in zip(mediums_gsm, mediums_bf, flute_types):
            rci = 0.06 * math.pow(bf, 0.8) * 0.88 # 12% loss for corrugation stress
            rct = (gsm / 1000.0) * rci * 15000.0
            takeup = takeup_map.get(ftype.upper(), 1.38)
            total_ect_n_m += rct * takeup

        # 4. McKee Formula: BCT (N) = 5.876 * ECT * sqrt(Perimeter * Caliper)
        bct_n = 5.876 * total_ect_n_m * math.sqrt(perimeter_m * caliper_m)
        bct_kgf_raw = bct_n / 9.80665

        predicted_bct_kgf = bct_kgf_raw * phi

        return {
            "ect_kn_m": round(total_ect_n_m / 1000.0, 2),
            "bct_raw_kgf": round(bct_kgf_raw, 2),
            "predicted_bct_kgf": round(predicted_bct_kgf, 2),
            "sealing_phi": phi
        }

In [10]:
# -------------------------------------------------------------------------
# Comprehensive Benchmark Validation Table
# -------------------------------------------------------------------------
def run_benchmark_validation(engine: UnifiedBCTModelEngine) -> pd.DataFrame:
    results = []
    for spec in BENCHMARK_SPECS:
        gsm = spec.total_gsm
        actual = spec.measured_bct_kgf

        # Empirical prediction
        emp_bct = engine.predict_empirical(gsm, phi=1.00)
        emp_err_pct = ((emp_bct - actual) / actual) * 100.0

        # First-principles prediction
        fp_res = engine.predict_first_principles(
            length_mm=spec.length_mm, width_mm=spec.width_mm, height_mm=spec.height_mm,
            liners_gsm=[spec.ply_structure[0], spec.ply_structure[2], spec.ply_structure[4]],
            liners_bf=[spec.bf_outer, spec.bf_mediums, spec.bf_outer],
            mediums_gsm=[spec.ply_structure[1], spec.ply_structure[3]],
            mediums_bf=[spec.bf_mediums, spec.bf_mediums],
            phi=1.00
        )
        fp_bct = fp_res["predicted_bct_kgf"]
        fp_err_pct = ((fp_bct - actual) / actual) * 100.0

        results.append({
            "Spec Description": spec.comment,
            "Total GSM": gsm,
            "Measured BCT (Kgf)": actual,
            "Empirical Model BCT (Kgf)": emp_bct,
            "Empirical Error (%)": f"{emp_err_pct:+.2f}%",
            "First-Principles BCT (Kgf)": fp_bct,
            "First-Principles Error (%)": f"{fp_err_pct:+.2f}%",
            "Board ECT (kN/m)": fp_res["ect_kn_m"]
        })
    return pd.DataFrame(results)

In [11]:
    # -------------------------------------------------------------------------
    # Paper & Financial Savings Engine
    # -------------------------------------------------------------------------
    def calculate_paper_financial_savings(self,
                                           baseline_gsm: float,
                                           target_bct_kgf: float,
                                           phi: float,
                                           annual_box_volume: int = 1000000,
                                           box_blank_area_sqm: float = 0.52,
                                           paper_cost_per_ton_usd: float = 650.0) -> Dict[str, float]:
        """Calculates material & financial savings when switching to glued sealing / optimized GSM."""
        optimized_gsm = self.invert_gsm_empirical(target_bct_kgf, phi=phi)
        gsm_delta = baseline_gsm - optimized_gsm
        savings_pct = (gsm_delta / baseline_gsm) * 100.0

        # Paper weight saved per box (grams -> kg)
        saved_kg_per_box = (gsm_delta * box_blank_area_sqm) / 1000.0
        annual_saved_tonnes = (saved_kg_per_box * annual_box_volume) / 1000.0
        annual_financial_savings_usd = annual_saved_tonnes * paper_cost_per_ton_usd

        return {
            "baseline_gsm": baseline_gsm,
            "optimized_gsm": optimized_gsm,
            "gsm_saved": round(gsm_delta, 1),
            "savings_pct": round(savings_pct, 2),
            "annual_saved_tonnes": round(annual_saved_tonnes, 2),
            "annual_savings_usd": round(annual_financial_savings_usd, 2)
        }

BENCHMARKS = BENCHMARK_SPECS


In [13]:
# -----------------------------------------------------------------------------
# SELF-TEST / CLI VERIFICATION
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    print("=================================================================")
    print(" UNIFIED BCT PREDICTION & GSM OPTIMIZATION ENGINE VERIFICATION")
    print("=================================================================\n")

    engine = UnifiedBCTModelEngine()

    print("--- [1. BENCHMARK COMPARISON TABLE (EMPIRICAL VS FIRST-PRINCIPLES)] ---")
    # Call run_benchmark_validation as a standalone function
    df_val = run_benchmark_validation(engine)
    print(df_val.to_string(index=False))

    print("\n--- [2. NOA KALANG TARGET (480 Kgf) SAVINGS ANALYSIS] ---")
    # Call calculate_paper_financial_savings as a standalone function, passing engine as the 'self' argument
    savings = calculate_paper_financial_savings(
        engine,
        baseline_gsm=880.0,
        target_bct_kgf=480.0,
        phi=1.209, # Glued @ 15mm
        annual_box_volume=1000000,
        box_blank_area_sqm=0.52,
        paper_cost_per_ton_usd=650.0
    )
    for k, v in savings.items():
        print(f"  • {k}: {v}")

    print("\n=================================================================")
    print(" VERIFICATION COMPLETE - ALL ENGINES OPERATIONAL")
    print("=================================================================")

 UNIFIED BCT PREDICTION & GSM OPTIMIZATION ENGINE VERIFICATION

--- [1. BENCHMARK COMPARISON TABLE (EMPIRICAL VS FIRST-PRINCIPLES)] ---
                      Spec Description  Total GSM  Measured BCT (Kgf)  Empirical Model BCT (Kgf) Empirical Error (%)  First-Principles BCT (Kgf) First-Principles Error (%)  Board ECT (kN/m)
  Original Spec Baseline (Min 800 Kgf)       1260               800.0                     792.62              -0.92%                      836.70                     +4.59%             14.36
        2023 First-Phase GSM Reduction       1050               600.0                     611.39              +1.90%                      700.00                    +16.67%             12.02
Optimized Spec (Tested for Noa Kalang)        880               480.0                     475.43              -0.95%                      592.43                    +23.42%             10.17

--- [2. NOA KALANG TARGET (480 Kgf) SAVINGS ANALYSIS] ---
  • baseline_gsm: 880.0
  • optimized_gsm: 77